<a href="https://colab.research.google.com/github/count-im/test/blob/main/LLM_Application/LLM04/LLM04_AutoGen_MultiAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM04 — AutoGen MultiAgent MovieAgent
**AIFFEL GenAI DAY5**

AutoGen을 활용한 멀티에이전트 시스템 구현
- 에이전트 3개: scenario_writer → scene_planner → storyboard_artist
- RoundRobinGroupChat 패턴
- 아이디어 한 줄 → 시나리오 → 콘티 → DALL-E 이미지 생성

In [3]:
# Cell 1 — 패키지 설치
!pip install autogen
!pip install -U "autogen-agentchat"
!pip install "autogen-ext[openai]"
!pip install openai

In [4]:
# Cell 2 — API Key 설정 (OPENAI_API_KEY 1개만)
import os
import getpass

def get_api_key(key_name: str) -> str:
    if os.environ.get(key_name):
        print(f"✅ {key_name}: 환경변수에서 로드")
        return os.environ[key_name]
    try:
        from google.colab import userdata
        value = userdata.get(key_name)
        if value:
            print(f"✅ {key_name}: Colab Secrets에서 로드")
            return value
    except Exception:
        pass
    print(f"🔑 {key_name}를 입력하세요:")
    value = getpass.getpass(f"{key_name}: ")
    return value

OPENAI_API_KEY = get_api_key("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("\n✅ API Key 설정 완료")

✅ OPENAI_API_KEY: 환경변수에서 로드

✅ API Key 설정 완료


In [5]:
# Cell 3 — Import
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
import openai

In [6]:
# Cell 4 — 모델 클라이언트 정의
# GPT-4o를 에이전트 LLM으로 사용 (원본 Gemini → GPT-4o 통일)
# API Key 1개로 에이전트 + 이미지 생성 모두 처리
openai_model_client = OpenAIChatCompletionClient(
    model="gpt-4o",
    api_key=OPENAI_API_KEY,
)
print("✅ GPT-4o 모델 클라이언트 준비 완료")

✅ GPT-4o 모델 클라이언트 준비 완료


In [7]:
# Cell 5 — 에이전트 시스템 메시지 정의
# 각 에이전트의 역할을 시스템 프롬프트로 정의 — 이것이 AutoGen 에이전트의 "성격"
# 마지막 에이전트(storyboard_artist)는 작업 완료 시 "TERMINATE" 출력 → 종료 조건 트리거

system_message_scenario_writer = '''
you are a scenario writer.
YOUR ANSWER MUST BE TEXT ONLY, NEVER CONTAIN ANY IMAGE.
You must write a creative story from given idea and characters.
Break down the scripted to 3~5 sub-scripts.
a sub-script includes *Plot* and *Involving Characters*, and you must write *Plot* and select *Involving Characters* from given character bank.
'''
description_scenario_writer = "a creative scenario writer"

system_message_scene_planner = '''
you are a scene planner.
YOUR ANSWER MUST BE TEXT ONLY, NEVER CONTAIN ANY IMAGE.
You must write a creative sub-script from given *Synopsis*.
Break down the Synopsis to 3~5 sub-scripts. a sub-script includes *Plot* and *Involving Characters*.
'''
description_scene_planner = "a creative scene planner"

system_message_storyboard_artist = '''
you are a storyboard artist.
YOUR ANSWER MUST BE TEXT ONLY, NEVER CONTAIN ANY IMAGE. YOUR OUTPUT MUST BE LESS THAN 3900 CHARACTERS INCLUDING BLANK.
You must evaluate the given script, and write a creative scene from given *sub-script*.
Break down the Sub-script to 3~5 Scenes.
a sub-script includes *Plot*, *Scene Description*, *Emotional tone*, *Visual style*, *Key props* and *Involving Characters* and *Cinematography Notes*
When you finish all scenes, write TERMINATE at the end.
'''
description_storyboard_artist = "a creative storyboard artist"

print("✅ 시스템 메시지 정의 완료")

✅ 시스템 메시지 정의 완료


In [8]:
# Cell 6 — 에이전트 3개 정의
# AssistantAgent = AutoGen의 기본 에이전트 단위
# name, model_client, description, system_message 4개가 핵심
scenario_writer_agent = AssistantAgent(
    name="scenario_writer_agent",
    model_client=openai_model_client,
    description=description_scenario_writer,
    system_message=system_message_scenario_writer,
)
scene_planner_agent = AssistantAgent(
    name="scene_planner_agent",
    model_client=openai_model_client,
    description=description_scene_planner,
    system_message=system_message_scene_planner,
)
storyboard_artist_agent = AssistantAgent(
    name="storyboard_artist_agent",
    model_client=openai_model_client,
    description=description_storyboard_artist,
    system_message=system_message_storyboard_artist,
)
print("✅ 에이전트 3개 정의 완료")

✅ 에이전트 3개 정의 완료


In [9]:
# Cell 7 — 아이디어 및 캐릭터 설정
# 여기만 바꾸면 완전히 다른 시나리오 생성 — 실습 핵심 커스터마이징 포인트
idea = '''
At the night, romeo wants to confess his mind to Juliette. Juliette is about to sleep at her room on 2nd story of her castle, and the room has a big window
'''
character_bank = '''
Romeo : Romeo is the very embodiment of romance. Bold and passionate, he wears his heart on his sleeve.
His love for Juliet is so deep, he would face death itself for her.
Yet, fate has cursed their love—his family and Juliet's are sworn enemies, locked in bitter hatred.

Juliet: Juliet fell for Romeo in an instant, as if struck by lightning.
She longs for a chance to speak with him, to share whispers beneath the moonlight.
But destiny is cruel—the blood feud between her family and Romeo's stands between them.
'''
print("✅ 아이디어 및 캐릭터 설정 완료")

✅ 아이디어 및 캐릭터 설정 완료


In [10]:
# Cell 8 — 그룹챗 + 종료 조건 정의
# RoundRobinGroupChat = 에이전트가 순서대로 돌아가며 대화
# TextMentionTermination = "TERMINATE" 텍스트 감지 시 자동 종료
# storyboard_artist의 시스템 메시지에 작업 완료 후 TERMINATE 출력 지시
termination = TextMentionTermination("TERMINATE")
group_chat = RoundRobinGroupChat(
    [scenario_writer_agent, scene_planner_agent, storyboard_artist_agent],
    termination_condition=termination
)
print("✅ 그룹챗 정의 완료")

✅ 그룹챗 정의 완료


In [11]:
# Cell 9 — 실행
# await Console() = 에이전트 대화를 실시간 스트리밍으로 출력
# Colab에서 async 환경 기본 지원 → await 직접 사용 가능
movie_task = f"write down a great scenario with given idea : {idea} and characters : {character_bank}"
result = await Console(group_chat.run_stream(task=movie_task))

---------- TextMessage (user) ----------
write down a great scenario with given idea : 
At the night, romeo wants to confess his mind to Juliette. Juliette is about to sleep at her room on 2nd story of her castle, and the room has a big window
 and characters : 
Romeo : Romeo is the very embodiment of romance. Bold and passionate, he wears his heart on his sleeve.
His love for Juliet is so deep, he would face death itself for her.
Yet, fate has cursed their love—his family and Juliet's are sworn enemies, locked in bitter hatred.

Juliet: Juliet fell for Romeo in an instant, as if struck by lightning.
She longs for a chance to speak with him, to share whispers beneath the moonlight.
But destiny is cruel—the blood feud between her family and Romeo's stands between them.

---------- TextMessage (scenario_writer_agent) ----------
**Sub-script 1: A Night of Anticipation**

*Plot*: As the midnight sky blankets the castle in darkness, Romeo stands beneath the towering walls, his heart racing 

In [12]:
# Cell 10 — storyboard_artist 출력 확인
# result.messages에서 storyboard_artist 출력만 필터링
for message in result.messages:
    if message.source == "storyboard_artist_agent":
        print(message.content)

**Scene 1: The Depths of Yearning**

*Scene Description*: The camera pans slowly from the dark, star-studded sky down to a shadowed figure standing resolutely beneath Juliet's window. Romeo's face is partially obscured by shadows, but the glow from the moon reveals his intense, hopeful eyes peering up towards the window. He's alone in the silent garden, adorned in his traditional cloak, embodying the quintessential romantic hero.

*Emotional Tone*: Deep longing and determination

*Visual Style*: High contrast between shadow and moonlight; a classic romantic ambiance strengthened by the gloomy yet hopeful night setting.

*Key Props*: Moonlit cloak, shadows, whispering leaves.

*Cinematography Notes*: Start with a wide aerial shot of the castle to showcase the setting, zooming into Romeo in the garden. Close-up on his face to showcase emotion.

---

**Scene 2: Juliet's Nocturnal Vigil**

*Scene Description*: Inside the warmly lit room of the castle, Juliet sits on the stone window ledge,

In [14]:
# Cell 11 — DALL-E 이미지 생성
# ⚠️ DALL-E 3 HD 1회 호출 = 약 $0.08 비용 발생

def get_subscripts(result):
    for message in result.messages:
        if message.source == "storyboard_artist_agent":
            return message.content

test_client = openai.OpenAI(api_key=OPENAI_API_KEY)

subscripts = get_subscripts(result)
base_prompt = "draw images describe subscripts in comic book format. Each frame should describes each sub-script and main characters of each plot must be shown in corresponding frame. Subscripts: "

# DALL-E 프롬프트 최대 4000자 제한 — 초과 시 자동 절단
max_content_len = 4000 - len(base_prompt)
prompt = base_prompt + subscripts[:max_content_len]

response = test_client.images.generate(
    model="dall-e-3",
    prompt=prompt,
    size="1024x1024",
    quality="hd",
    n=1,
)
print("✅ 이미지 생성 완료")
print("이미지 URL:")
print(response.data[0].url)

✅ 이미지 생성 완료
이미지 URL:
https://oaidalleapiprodscus.blob.core.windows.net/private/org-ZVmiEUMIhzaQFCyKy7RQcWOR/user-N4PGQR6TRbsUfOveUqzAXu27/img-91s1oyYvVWCtY1hB64A3g8OU.png?st=2026-03-19T02%3A59%3A40Z&se=2026-03-19T04%3A59%3A40Z&sp=r&sv=2026-02-06&sr=b&rscd=inline&rsct=image/png&skoid=32836cae-d25f-4fe9-827b-1c8c59c442cc&sktid=a48cca56-e6da-484e-a814-9c849652bcb3&skt=2026-03-19T03%3A44%3A31Z&ske=2026-03-20T03%3A44%3A31Z&sks=b&skv=2026-02-06&sig=kBT%2BzxuA0QUDmiC8kUV6L/wlWbObC4rziImVOGaZEes%3D


In [ ]:
# Cell 12 — 결과 확인
response